In [ ]:
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

In [ ]:
#Initial conditions
G=6.67*10**(-11)
dt=10
N=36500


In [ ]:
#Separation between ij
def sep_ij(r1,r2):
    a=(r1[0]-r2[0])**2+(r1[1]-r2[1])**2+(r1[2]-r2[2])**2
    return np.sqrt(a)

In [ ]:
def acc(v):
    for i in range(0,1):
        dv=v[i+1]-v[i]
    return dv/dt

In [ ]:
def force_ij(acc,m1,m2):
    d

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
#Constants
G=6.67*10**(-11)
eps=0.1

#Initial conditions
v=np.zeros((3,2*N))
r1=np.zeros((3,2*N))
r2=np.zeros((3,2*N))
r3=np.zeros((3,2*N))
r1[0]=([1,1,1])
r2[0]=([2,2,2])
r3[0]=([3,3,3])
masses=[0,0,0]
#introduce the loop
#define functions
def sep_ij(r1,r2):
    for i in range (0,2*N):
        a=(r1[i][0]-r2[i][0])**2+(r1[i][1]-r2[i][1])**2+(r1[i][2]-r2[i][2])**2
        return np.sqrt(a)
def force(r1,r2,masses):
    for i in range (0,2*N):
        force=[]
        for j in range (0,2*N):
            if i==j:
                pass
            else:
                f=G*(masses[i]*masses[j]*(r1-r2)/((sep(r1,r2)**2+eps**2)**(3/2)))
                force.append(f)
            return force
def acc(r1,r2,masses):
    a=[]
    for i in range (0,2*N):
        ac=force(r1,r2,masses)/masses[i]
        a=a+[ac]
    return a
for i in range(0,2*N):
    v[2*i]=v[i]+acc(r[i],r[i+1],masses)*dt

for i in range(0,2*N):
    r[2*i]=r[i]+v[i]*dt

v=v[1::2]
r=r[1::2]

#figure out how to plot animations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Constants
G = 6.67*10**(-11)          # N-body units (6.67e-11 would need huge masses)
eps = 0.        # softening length
dt = 864000
steps = 3650
N = 3

masses = np.array([1.9889*10**30, 5.9742*10**24, 1.8986*10**27])

# Initial conditions: pos[i] = (x, y, z) of body i
pos = np.array([[0. , 0. , 0. ],
                [1.4960*10**11, 0., 0.],
                [7.7854*10**11, 0., 0.]])
pos += np.random.default_rng(0).normal(scale=0.3, size=pos.shape)  # break the collinear setup
vel = np.array([[ 0.0,  0.0,  0.0],   # body 0
                [ 0.0, 29785., 0.0],   # body 1
                [ 0.0,  13069., 0.0]])  # body 2

def acceleration(pos, masses):
    # dr[i, j] = pos[j] - pos[i]  -> shape (N, N, 3)
    dr = pos[None, :, :] - pos[:, None, :]
    dist2 = np.sum(dr**2, axis=2) + eps**2
    inv_d3 = dist2**(-1.5)
    np.fill_diagonal(inv_d3, 0.0)           # no self-force
    # a_i = G * sum_j m_j * dr_ij / (|dr_ij|^2 + eps^2)^(3/2)
    return G * np.sum(masses[None, :, None] * dr * inv_d3[:, :, None], axis=1)

# Leapfrog (kick-drift-kick): much better energy behaviour than plain Euler
history = np.zeros((steps, N, 3))
a = acceleration(pos, masses)
ke = np.zeros(steps)
gpe = np.zeros(steps)
vel -= np.average(vel, axis=0, weights=masses)
pos -= np.average(pos, axis=0, weights=masses)
def energies(pos, vel, masses):
    ke = 0.5 * np.sum(masses * np.sum(vel**2, axis=1))
    dr = pos[None, :, :] - pos[:, None, :]
    dist = np.sqrt(np.sum(dr**2, axis=2) + eps**2)   # same softening as the force
    iu = np.triu_indices(N, k=1)                      # each pair counted once
    gpe = -G * np.sum(masses[iu[0]] * masses[iu[1]] / dist[iu])
    return ke, gpe

for n in range(steps):
    history[n] = pos
    ke[n], gpe[n] = energies(pos, vel, masses)
    vel += 0.5 * dt * a
    pos += dt * vel
    a = acceleration(pos, masses)
    vel += 0.5 * dt * a

fig = plt.figure()
ax = fig.add_subplot(projection="3d")
lo, hi = history.min() - 0.5, history.max() + 0.5
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_zlim(lo, hi)

dots = [ax.plot([], [], [], "o")[0] for _ in range(N)]
trails = [ax.plot([], [], [], "-", lw=0.8)[0] for _ in range(N)]



def update(n):
    for i in range(N):
        x, y, z = history[:n+1, i].T
        trails[i].set_data(x, y); trails[i].set_3d_properties(z)
        dots[i].set_data(x[-1:], y[-1:]); dots[i].set_3d_properties(z[-1:])
    return dots + trails

anim = FuncAnimation(fig, update, frames=range(0, steps, 5), interval=20, blit=False)
#plt.show()
# anim.save("nbody.gif", writer="pillow")   # to save instead

In [ ]:
from IPython.display import HTML
anim.embed_limit=1024
HTML(anim.to_jshtml())

In [ ]:
plt.rcParams["font.size"] = 14
plt.plot(ke, label="Kinetic")
plt.plot(gpe, label="Potential")
plt.plot(2*ke + gpe, label="Virial")
plt.xlabel("Step")
plt.ylabel("Energy")
plt.legend()
plt.grid(alpha=0.3)
ax=plt.gca()
#plt.ylim(-1.*10**33,1.*10**33)
#plt.xlim(0,500)
axins = ax.inset_axes([0.08, 0.15, 0.4, 0.4])  # [left, bottom, width, height] in axes fraction
axins.plot(2*ke+gpe, color='green')
axins.grid(alpha=0.3)
axins.set_xlim(0, 500)
axins.set_ylim(-0.8*10**34, 0.8*10**34)
ax.indicate_inset_zoom(axins)
rect, connectors = ax.indicate_inset_zoom(axins, edgecolor="black", alpha=1.0)

plt.savefig('energies1.png',dpi=150)

In [ ]:
1-gpe[-1]/gpe[0]


In [ ]:
1-(gpe[-1]+ke[-1])/((gpe[0]+ke[0]))

In [ ]:
np.abs((pos[0,-1]/pos[0,0]))

In [ ]:
[print(pos) for pos in history]
#plt.plot(pos[0][0],pos[0][1])

In [ ]:
plt.plot(pos[0][1])

In [ ]:
for i in range(N):
    plt.plot(history[:, i, 0], history[:, i, 1], color=colors[i], lw=1, label=f"Body {i}")

    plt.legend()
    #plt.scatter(history[0, i, 0], history[0, i, 1], color=colors[i], marker="o")
    #plt.scatter(history[-1, i, 0], history[-1, i, 1], color=colors[i], marker="x")

In [ ]:
plt.subplot()
plt.figure(figsize=(6,6))
plt.plot(history[:, 0, 0], history[:, 0, 1], lw=1, label="Sun")
plt.plot(history[:, 1, 0], history[:, 1, 1], lw=1, label="Earth")
plt.plot(history[:, 2, 0], history[:, 2, 1], lw=1, label="Jupiter")

plt.xlabel("Distance in x-direction (m)")
plt.ylabel("Distance in y-direction (m)")
plt.legend()
ax=plt.gca()
axins = ax.inset_axes([0.35, 0.67, 0.3, 0.3])  # [left, bottom, width, height] in axes fraction
axins.plot(history[:, 1, 0], history[:, 1, 1], lw=0.03, color='orange', )
axins.grid(alpha=0.3)
axins.set_xlim(-0.2*10**11, .2*10**11)
axins.set_ylim(1.3*10**11, 1.7*10**11)
axins.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
ax.indicate_inset_zoom(axins)
rect, connectors = ax.indicate_inset_zoom(axins, edgecolor="black", alpha=1.0)

plt.savefig("orbits.png", dpi=150)


In [ ]:
x=history[:, 0, 0]-history[:, 1,0]
y=history[:, 0, 1]-history[:, 1,1]
sep=np.sqrt(x**2+y**2)

In [ ]:
plt.plot(sep)
plt.ylim(1.49*10**11, 1.53*10**11)